# Image Moments (immoments)

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/image_analysis/moments.ipynb)

Collapse an image cube along one axis into a set of **moment maps** -- AstroVIPER's
reimplementation of CASA's `immoments` task. The compute runs as a GraphVIPER map
graph: each node task loads only its own chunk of the input image from disk,
computes every requested moment for that chunk with a memory-frugal streaming
algorithm, and writes its own slice of the output Zarr store in parallel.

---

## Assumptions and Background

**AstroVIPER axis nomenclature** (vs CASA's `axis` parameter):

| CASA | AstroVIPER `moment_axis` |
| --- | --- |
| ra | `l` |
| dec | `m` |
| spectral | `frequency` |
| stokes | `polarization` |
| -- | `time` |

**Available moments** (canonical names, with the CASA `immoments` integer codes also accepted):

| Name | CASA code | Definition |
| --- | --- | --- |
| `mean` | -1 | mean value of the profile |
| `integrated` | 0 | `sum(I * delta_v)` (delta_v = per-plane coordinate width) |
| `weighted_coord` | 1 | `sum(I*v)/sum(I)` (e.g. velocity field) |
| `weighted_dispersion_coord` | 2 | `sqrt(sum(I*v^2)/sum(I) - m1^2)` |
| `median` | 3 | median of the profile |
| `median_coord` | 4 | coordinate where the cumulative profile crosses 50% |
| `standard_deviation` | 5 | standard deviation about the profile mean |
| `rms` | 6 | root mean square of the profile |
| `abs_mean_dev` | 7 | mean absolute deviation |
| `maximum` / `maximum_coord` | 8 / 9 | profile maximum and its coordinate |
| `minimum` / `minimum_coord` | 10 / 11 | profile minimum and its coordinate |

Key points:

- The **moment axis is never used for parallelism** (every moment needs the full
  axis). The graph is chunked along `parallel_axis` instead: `frequency` by
  default, or `m` when the moment axis *is* `frequency`.
- Coordinate-valued moments are expressed in the **native units of the moment
  axis** (Hz for `frequency`, rad for `l`/`m`); CASA instead converts the
  spectral axis to velocity.
- Pixels are excluded if they are NaN, outside `include_pixel_range` (or inside
  `exclude_pixel_range`), or `False` in the data group's mask (`use_mask=True`).
- The output is a **single Zarr image store** with one `SKY_MOMENT_<NAME>` data
  variable and one `moment_<name>` data group per requested moment; the moment
  axis is kept as a degenerate size-1 dimension whose numeric coordinates are
  the mean of the input coordinates.


---
## Pseudo Code

```
distributed_applications/image_analysis/moments(input_image_store, moments_image_store, ...)
    open input image lazily; resolve sky/mask variables from the data group
    choose parallel_axis (never the moment axis); estimate per-chunk memory
    create the empty output image on disk (moment axis collapsed to size 1)
    pre-allocate the SKY_MOMENT_* Zarr arrays (NaN filled)
    map over parallel_axis chunks:
        node_tasks/image_analysis/moments(chunk)
            load only this chunk's sky (+ mask) variables
            processing_functions/image_analysis/moments(chunk_xds)
                stream plane-by-plane along the moment axis (map-sized accumulators)
            write this chunk's slice of every moment map to the Zarr store
    consolidate Zarr metadata; return the per-task timing DataFrame
```

---
## API


In [ ]:
from astroviper.distributed_applications.image_analysis import moments

moments?

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401 -- availability probe for the pip-install fallback

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))

## Example 1: Moments of a synthetic spectral-line cube

We build a small synthetic cube containing a Gaussian spectral line whose center
frequency drifts across the image (a mock velocity gradient), write it to a Zarr
store, and compute moments over the `frequency` axis.

In [ ]:
import numpy as np
import xarray as xr
from xradio.image import make_empty_sky_image, write_image

rad_per_arcsec = np.pi / 180 / 3600
n_frequency, n_l, n_m = 32, 64, 64
frequency = np.linspace(1.40e9, 1.42e9, n_frequency)

img_xds = make_empty_sky_image(
    phase_center=[0.6, -0.2],
    image_size=[n_l, n_m],
    cell_size=[15 * rad_per_arcsec, 15 * rad_per_arcsec],
    frequency_coords=frequency,
    pol_coords=["I"],
    time_coords=[0],
)

# Gaussian source in the image plane x Gaussian line in frequency, with the
# line center drifting linearly along l (a mock velocity gradient).
l_idx, m_idx = np.meshgrid(np.arange(n_l), np.arange(n_m), indexing="ij")
spatial = np.exp(-(((l_idx - n_l / 2) ** 2 + (m_idx - n_m / 2) ** 2) / (2 * 8.0**2)))
line_center = 1.41e9 + 4e6 * (l_idx - n_l / 2) / n_l  # Hz, drifts with l
line_width = 2.5e6  # Hz
profile = np.exp(
    -((frequency[:, None, None] - line_center[None]) ** 2) / (2 * line_width**2)
)
rng = np.random.default_rng(1)
sky = (spatial[None] * profile + rng.normal(0, 0.02, profile.shape)).astype(np.float32)

img_xds["SKY"] = xr.DataArray(
    sky[None, :, None], dims=["time", "frequency", "polarization", "l", "m"]
)
img_xds["SKY"].attrs["units"] = "Jy/beam"
img_xds.attrs["data_groups"] = {"base": {"sky": "SKY"}}

input_image_store = "moments_demo_input.img.zarr"
write_image(img_xds, imagename=input_image_store, out_format="zarr", overwrite=True)
print(img_xds.SKY.shape)

In [ ]:
moments_image_store = "moments_demo.img.zarr"
timing_df = moments(
    input_image_store=input_image_store,
    moments_image_store=moments_image_store,
    moments=["integrated", "weighted_coord", "weighted_dispersion_coord", "maximum"],
    moment_axis="frequency",
    # Restrict the intensity-weighted moments to significant emission
    # (recommended for weighted_coord / weighted_dispersion_coord).
    include_pixel_range=[0.1, 1e9],
    overwrite=True,
)
timing_df

### Inspect the output

The output store holds one `SKY_MOMENT_<NAME>` data variable per moment, the
`frequency` axis collapsed to size 1 (its coordinate is the mean input
frequency), and one `moment_<name>` data group per moment.

In [ ]:
from xradio.image import load_image

moments_xds = load_image(moments_image_store)
print(list(moments_xds.data_vars))
print(dict(moments_xds.sizes))
print(list(moments_xds.attrs["data_groups"].keys()))
moments_xds

In [ ]:
# %matplotlib widget  # uncomment for interactive plots (not for headless runs)
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
maps = [
    ("SKY_MOMENT_INTEGRATED", "integrated (Jy/beam.Hz)"),
    ("SKY_MOMENT_WEIGHTED_COORD", "weighted coord (Hz)"),
    ("SKY_MOMENT_WEIGHTED_DISPERSION_COORD", "dispersion (Hz)"),
]
for ax, (name, title) in zip(axes, maps, strict=False):
    image = moments_xds[name].isel(time=0, frequency=0, polarization=0).values
    im = ax.imshow(image, origin="lower")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.show()

The `weighted_coord` map recovers the linear drift of the line center along `l`
(where there is enough emission to pass `include_pixel_range`), and the
dispersion map recovers the (constant) line width where the line is bright.

## Example 2: CASA integer codes and other moment axes

CASA `immoments` codes work directly, and the moment axis can be any of `l`,
`m`, `frequency`, `polarization` or `time` -- the graph automatically
parallelizes over `frequency` when the moment axis is spatial.

In [ ]:
timing_df = moments(
    input_image_store=input_image_store,
    moments_image_store="moments_demo_m_axis.img.zarr",
    moments=[-1, 8],  # CASA codes: mean, maximum
    moment_axis="m",  # parallelizes over frequency
    overwrite=True,
)
m_axis_xds = load_image("moments_demo_m_axis.img.zarr")
print(list(m_axis_xds.data_vars), dict(m_axis_xds.sizes))

---
## Notes

- The moment axis can never be the parallel axis; `parallel_axis` must be one
  of `frequency`, `m` or `time` (the `l` coordinate decreases, which GraphVIPER's
  chunk interpolation does not support, and `polarization` is non-numeric).
- Coordinate-valued moments are in the native axis units (Hz, rad, or plane
  index for `polarization`); convert to velocity downstream if needed.
- `median` is the only moment that needs a full-axis working copy per chunk;
  everything else streams plane-by-plane. The driver's automatic chunk count
  accounts for this.

## Questions

- Should coordinate-valued spectral moments optionally be expressed in velocity
  (matching CASA), given a rest frequency?
